# Segment-Specific Demand Modeling

This notebook contains my demand-modeling work for the DealMakers dynamic pricing project.

Customers are divided into eight segments using median splits on three covariates. A separate Logistic Regression model is trained for each segment to estimate purchase probability as a function of customer characteristics and price.

The trained segment-specific models are serialized into `8_models_dict.pkl` and later used by my dynamic pricing agent.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression



In [21]:
import pickle

## 1. Data Loading

In [2]:
train_pricing_decisions = pd.read_csv('train_prices_decisions_2025.csv')
test_user_info = pd.read_csv('test_user_info_2025.csv')

In [3]:
train_pricing_decisions.head()

,user_index,Covariate1,Covariate2,Covariate3,price_item,item_bought
0,0,1.396069,0.228525,9.790878,0.010000,True
1,1,5.201819,2.212083,10.549586,50.563709,True
2,2,5.471765,0.568564,12.706814,82.305243,True
3,3,3.002355,1.185335,5.513210,38.611238,True
4,4,1.337119,1.374958,6.298567,86.705523,False


## 2. Customer Segmentation

Customers are divided into eight segments using median thresholds on the three covariates.

In [25]:
train_pricing_decisions["Covariate1"].median()

np.float64(2.7193025761078644)

In [26]:
train_pricing_decisions["Covariate2"].median()

np.float64(2.7215555543935457)

In [31]:
train_pricing_decisions["Covariate3"].median()

np.float64(7.262601783583493)

In [ ]:
cov1_median = train_pricing_decisions["Covariate1"].median()
cov2_median = train_pricing_decisions["Covariate2"].median()
cov3_median = train_pricing_decisions["Covariate3"].median()

In [ ]:
def add_binary_features(df):
    df = df.copy()
    df["cov1_high"] = (df["Covariate1"] > cov1_median).astype(int)
    df["cov2_high"] = (df["Covariate2"] > cov2_median).astype(int)
    df["cov3_high"] = (df["Covariate3"] > cov3_median).astype(int)
    return df

train_df = add_binary_features(train_pricing_decisions)
test_df = add_binary_features(test_user_info)

### Build the Eight Training Segments

In [7]:
segments = {}
for c1 in [0,1]:
    for c2 in [0,1]:
        for c3 in [0,1]:
            key = (c1,c2,c3)
            segments[key] = train_df[
                (train_df["cov1_high"]==c1) &
                (train_df["cov2_high"]==c2) &
                (train_df["cov3_high"]==c3)
            ]


In [ ]:
test_segments = {}
for c1 in [0,1]:
    for c2 in [0,1]:
        for c3 in [0,1]:
            key = (c1,c2,c3)
            test_segments[key] = test_df[
                (test_df["cov1_high"]==c1) &
                (test_df["cov2_high"]==c2) &
                (test_df["cov3_high"]==c3)
            ]

## 3. Train Segment-Specific Logistic Regression Models

In [11]:
models = {}

for key, seg_df in segments.items():
    model = LogisticRegression()
    model.fit(
        seg_df[["Covariate1","Covariate2","Covariate3","price_item"]].values,
        seg_df["item_bought"].values
    )
    models[key] = model

## 4. Personalized Price Optimization

For each customer, evaluate a grid of candidate prices and choose the price that maximizes immediate expected revenue:

`purchase_probability × price`

In [12]:
def compute_optimal_prices(
    model,
    user_covariates,
    price_min=0.1,
    price_max=120,
    num_prices=200,
    scaler=None
):

    price_grid = np.linspace(price_min, price_max, num_prices)
    n_users = user_covariates.shape[0]

    prices = price_grid.reshape(1, -1).repeat(n_users, axis=0)

    cov_expanded = np.repeat(user_covariates, num_prices, axis=0)

    prices_flat = prices.flatten().reshape(-1, 1)

    X_all = np.hstack([cov_expanded, prices_flat])

    if scaler is not None:
        X_all = scaler.transform(X_all)

    probs = model.predict_proba(X_all)[:, 1]

    probs_matrix = probs.reshape(n_users, num_prices)

    revenues = probs_matrix * price_grid

    best_idx = revenues.argmax(axis=1)
    best_prices = price_grid[best_idx]
    best_revenues = revenues[np.arange(n_users), best_idx]

    return best_prices, best_revenues

## 5. Generate Segment-Specific Pricing Decisions

In [13]:
results_list = []

for key, seg_df in test_segments.items():
    
    if len(seg_df) == 0:
        continue

    model = models[key]

    X_seg = seg_df[["Covariate1","Covariate2","Covariate3"]].values

    best_prices, best_revenues = compute_optimal_prices(
        model=model,
        user_covariates=X_seg
    )

    tmp = seg_df.copy()
    tmp["optimal_price"] = best_prices
    tmp["optimal_revenue"] = best_revenues

    results_list.append(tmp)


In [14]:
final_df = pd.concat(results_list, axis=0)

In [15]:
final_df = final_df.sort_values("user_index").reset_index(drop=True)

In [16]:
final_df

,user_index,Covariate1,Covariate2,Covariate3,cov1_high,cov2_high,cov3_high,optimal_price,optimal_revenue
0,50000,0.584638,5.706175,9.668797,0,1,1,7.932663,8.656892e-02
1,50001,3.548108,7.374092,5.104997,1,1,0,4.317588,7.394317e-07
2,50002,1.713915,4.704735,8.012817,0,1,1,4.317588,4.851462e-01
3,50003,5.767208,1.807943,6.023578,1,0,0,54.928643,5.113582e+01
4,50004,1.021004,2.912415,11.899356,0,1,1,36.853266,2.878030e+01
...,...,...,...,...,...,...,...,...,...
49995,99995,0.915252,0.174688,4.327159,0,0,0,74.811558,7.099339e+01
49996,99996,8.611839,0.893256,4.429134,1,0,0,120.000000,1.195929e+02
49997,99997,1.222712,3.669642,4.454837,0,1,0,11.547739,7.129495e+00
49998,99998,1.160867,3.250831,5.595728,0,1,0,17.572864,9.336412e+00


In [17]:
final_output = final_df[['user_index', 'optimal_price', 'optimal_revenue']].rename(
    columns={
        'user_index': 'user_index',
        'optimal_price': 'price_item',
        'optimal_revenue': 'expected_revenue'
    }
)


In [18]:
final_output

,user_index,price_item,expected_revenue
0,50000,7.932663,8.656892e-02
1,50001,4.317588,7.394317e-07
2,50002,4.317588,4.851462e-01
3,50003,54.928643,5.113582e+01
4,50004,36.853266,2.878030e+01
...,...,...,...
49995,99995,74.811558,7.099339e+01
49996,99996,120.000000,1.195929e+02
49997,99997,11.547739,7.129495e+00
49998,99998,17.572864,9.336412e+00


In [19]:
print(f"Revenue: {final_output['expected_revenue'].sum()}")

Revenue: 2208764.9594354923


In [20]:
final_output.to_csv("part1_static_prices_submission.csv", index=False)
final_output.head()

,user_index,price_item,expected_revenue
0,50000,7.932663,8.656892e-02
1,50001,4.317588,7.394317e-07
2,50002,4.317588,4.851462e-01
3,50003,54.928643,5.113582e+01
4,50004,36.853266,2.878030e+01


## 6. Save Trained Demand Models

Serialize the eight segment-specific Logistic Regression models for use by the downstream pricing agent.

In [23]:
filename = '8_models_dict.pkl'

In [24]:
with open(filename, 'wb') as file:
    pickle.dump(models, file)